<a href="https://colab.research.google.com/github/Perez-Alvaro/uber-lyft-dataset/blob/main/Contenido%20Entrega%203/TPI_Ciencia_de_Datos_Sprint3_Modelado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚖 Proyecto Integrador – Ciencia de Datos (UTN-FRC)
## **Sprint 3: Modelado Predictivo, Balanceo de Clases y Evaluación Multimodelo**
**Curso:** 5K1 | **Grupo 11** | **Docente:** Ing. Marisa del Carmen Callejas  
**Integrantes:**
- Alvaro Perez (97986)
- Juan Ignacio Cremona (95789)
- Ignacio Gil (407114)
- Federico Gon (94470)
- Sofía Medina (88655)
- Facundo Dagnino Dailly (94307)

---

### 🎯 Objetivos y Directivas de la Cátedra en este Sprint:
1. **Auditoría de Distribuciones:** Diagnosticar la forma funcional del target continuo (`price`) y el severo desbalance de la tarifa dinámica (`has_surge`).
2. **Laboratorio de Balanceo de Datos:** Probar y comparar técnicas de submuestreo (*Undersampling*), sobremuestreo sintético (*SMOTE*) y ponderación algorítmica (*Cost-sensitive / Class-Weight*).
3. **Modelado de Regresión (`price`):** Comparar **Regresión Lineal** (directa y logarítmica) vs. **Árbol de Regresión**, evaluando bondad de ajuste ($R^2$), error cuadrático medio (RMSE) y residuos.
4. **Modelado de Clasificación Binaria en Lyft (`has_surge`):** Entrenar **Regresión Logística** y **Árbol de Decisión**, evaluando con **Matrices de Confusión**, Recall, Precision y PR-AUC bajo estricta prevención de *Data Leakage*.
5. **Modelado de Clasificación Multiclase en Uber (`service_tier`):** Explotar la arquitectura de *Upfront Pricing* para inferir la gama del viaje a partir de la distancia y el costo por milla, analizando la **Matriz de Confusión Multiclase ($5 \times 5$)**.


---
## 📦 0. Configuración del Entorno y Carga de Datos
Instalación de librerías requeridas y carga automática del dataset limpio curado en la Entrega 2.


In [ ]:
# Instalación de librerías especializadas si se ejecuta en Google Colab
!pip install -q imbalanced-learn scikit-learn seaborn matplotlib

import os
import sys
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    confusion_matrix, classification_report,
    roc_auc_score, precision_recall_curve, auc,
    f1_score, recall_score, precision_score, accuracy_score
)
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

print("[OK] Entorno y dependencias configuradas con éxito.")


In [ ]:
# Localización o descarga del dataset curado (dataset_nueva_columna.zip)
POSIBLES_RUTAS = [
    "Contenido Entrega 2/DataSet Segunda Entrega Sin Codificar/dataset_nueva_columna.zip",
    "../Contenido Entrega 2/DataSet Segunda Entrega Sin Codificar/dataset_nueva_columna.zip",
    "dataset_nueva_columna.zip"
]

data_path = None
for r in POSIBLES_RUTAS:
    if os.path.exists(r):
        data_path = r
        break

if data_path is None:
    print("[*] Descargando dataset desde el repositorio oficial de GitHub...")
    url = "https://raw.githubusercontent.com/Perez-Alvaro/uber-lyft-dataset/main/Contenido%20Entrega%202/DataSet%20Segunda%20Entrega%20Sin%20Codificar/dataset_nueva_columna.zip"
    data_path = "dataset_nueva_columna.zip"
    urllib.request.urlretrieve(url, data_path)

print(f"[+] Cargando dataset desde: {data_path}")
df = pd.read_csv(data_path)
print(f"[✓] Dataset cargado: {df.shape[0]:,} registros x {df.shape[1]} columnas.")
df.head(3)


---
## 📊 1. Auditoría de Distribuciones y Desbalance de Clases
Evaluamos el comportamiento de las dos variables objetivo principales:
- **`price` (Target Continuo):** Análisis de asimetría positiva (*skewness*) y justificación de la transformación logarítmica.
- **`has_surge` (Target Binario):** Demostración empírica del desbalance de clases (Lyft vs. Uber).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Distribución de Precios (Original vs Log)
sns.histplot(df['price'], bins=40, kde=True, color='#1f77b4', ax=axes[0])
axes[0].set_title(f"Distribución de price (Sesgo positivo = {df['price'].skew():.2f})")
axes[0].set_xlabel("Precio ($ USD)")
axes[0].set_ylabel("Frecuencia")

# 2. Distribución de has_surge en Lyft (sin Shared)
df_lyft_valid = df[(df['cab_type'] == 'Lyft') & (df['name'] != 'Shared')]
surge_counts = df_lyft_valid['has_surge'].value_counts()
surge_pcts = df_lyft_valid['has_surge'].value_counts(normalize=True) * 100

axes[1].bar(['Sin Surge (0)', 'Con Surge (1)'], surge_counts.values, color=['#4a90e2', '#d9534f'], width=0.5)
for i, (cnt, pct) in enumerate(zip(surge_counts.values, surge_pcts.values)):
    axes[1].text(i, cnt + 3000, f"{cnt:,}\n({pct:.2f}%)", ha='center', fontweight='bold')

axes[1].set_title("Desbalance Crítico de has_surge en Lyft (Sin Shared)")
axes[1].set_ylabel("Cantidad de Cotizaciones")
axes[1].set_ylim(0, surge_counts.max() * 1.15)

plt.tight_layout()
plt.show()

print(f"Total cotizaciones Lyft analizadas: {len(df_lyft_valid):,}")
print(f"Porcentaje de viajes con recargo dinámico: {surge_pcts[1]:.2f}%")
print(f"Ratio de desbalance negativo:positivo = {surge_counts[0]/surge_counts[1]:.1f} a 1")


---
## 📈 2. Módulo de Regresión: Modelado de Tarifa (`price`)
Cumpliendo con la consigna de comparar un **modelo lineal** y un **árbol**:
1. **Regresión Lineal Simple / OLS:** Entrenada sobre el precio directo.
2. **Regresión Lineal Logarítmica:** Entrenada sobre $\log(\text{price})$ para estabilizar la varianza residual ante outliers de gama alta.
3. **Árbol de Decisión Regresor (`DecisionTreeRegressor`):** Captura interacciones no lineales entre distancia, franja horaria y nivel de servicio.


In [ ]:
# Selección de atributos predictivos para regresión
features_reg = [
    'distance', 'straight_line_miles', 'service_tier', 'cab_type',
    'time_slot', 'is_rush_hour', 'is_weekend', 'temperature', 'humidity', 'windSpeed'
]

# Preparación de variables dummy (drop_first=True para evitar multicolinealidad estricta)
X_reg = pd.get_dummies(df[features_reg], drop_first=True, dtype=float)
y_reg = df['price'].values
y_reg_log = np.log(y_reg)

# Partición Train / Test (80% entrenamiento, 20% prueba)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
_, _, y_tr_r_log, y_te_r_log = train_test_split(X_reg, y_reg_log, test_size=0.2, random_state=42)

# 1. Regresión Lineal Directa
reg_lineal = LinearRegression()
reg_lineal.fit(X_tr_r, y_tr_r)
y_pred_rl = reg_lineal.predict(X_te_r)
r2_rl = r2_score(y_te_r, y_pred_rl)
rmse_rl = np.sqrt(mean_squared_error(y_te_r, y_pred_rl))
mae_rl = mean_absolute_error(y_te_r, y_pred_rl)

# 2. Regresión Lineal Logarítmica
reg_log = LinearRegression()
reg_log.fit(X_tr_r, y_tr_r_log)
y_pred_rlog = np.exp(reg_log.predict(X_te_r)) # Exponencial inversa
r2_rlog = r2_score(y_te_r, y_pred_rlog)
rmse_rlog = np.sqrt(mean_squared_error(y_te_r, y_pred_rlog))
mae_rlog = mean_absolute_error(y_te_r, y_pred_rlog)

# 3. Árbol de Regresión (Control de profundidad para evitar sobreajuste)
arbol_reg = DecisionTreeRegressor(max_depth=12, random_state=42)
arbol_reg.fit(X_tr_r, y_tr_r)
y_pred_arbol = arbol_reg.predict(X_te_r)
r2_arbol = r2_score(y_te_r, y_pred_arbol)
rmse_arbol = np.sqrt(mean_squared_error(y_te_r, y_pred_arbol))
mae_arbol = mean_absolute_error(y_te_r, y_pred_arbol)

# Tabla comparativa
tabla_reg = pd.DataFrame({
    'Modelo': ['Regresión Lineal (Directa)', 'Regresión Lineal (Logarítmica)', 'Árbol Regresor (depth=12)'],
    'R² (Bondad de Ajuste)': [f"{r2_rl:.4f}", f"{r2_rlog:.4f}", f"{r2_arbol:.4f}"],
    'RMSE ($ USD)': [f"${rmse_rl:.2f}", f"${rmse_rlog:.2f}", f"${rmse_arbol:.2f}"],
    'MAE ($ USD)': [f"${mae_rl:.2f}", f"${mae_rlog:.2f}", f"${mae_arbol:.2f}"]
})
print("=== RESULTADOS COMPARATIVOS: MODELOS DE REGRESIÓN ===")
display(tabla_reg)


In [ ]:
# Visualización: Valores Reales vs. Predicciones y Residuos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

modelos_reg_plot = [
    ("Regresión Lineal", y_pred_rl, '#1f77b4', r2_rl),
    ("Regresión Logarítmica", y_pred_rlog, '#ff7f0e', r2_rlog),
    ("Árbol de Decisión", y_pred_arbol, '#2ca02c', r2_arbol)
]

for ax, (nombre, y_p, col, r2_val) in zip(axes, modelos_reg_plot):
    # Submuestra de 2,000 puntos para visualización nítida
    idx_sample = np.random.choice(len(y_te_r), size=2000, replace=False)
    ax.scatter(y_te_r[idx_sample], y_p[idx_sample], alpha=0.25, color=col, s=15)
    ax.plot([0, y_te_r.max()], [0, y_te_r.max()], 'r--', lw=2, label="Ajuste Ideal")
    ax.set_title(f"{nombre}\nR² = {r2_val:.4f}")
    ax.set_xlabel("Precio Real ($)")
    ax.set_ylabel("Precio Predicho ($)")
    ax.legend(loc="upper left")

plt.tight_layout()
plt.show()


---
## ⚖️ 3. Módulo de Clasificación Binaria (Lyft `has_surge`) y Laboratorio de Balanceo
En este módulo abordamos el **desbalance extremo de clases** (91.8% vs. 8.2%) aplicando las técnicas requeridas por la cátedra:
1. **Desbalanceado (Línea Base):** Entrenar con la distribución real sin intervenir.
2. **Undersampling (Submuestreo):** Eliminar registros mayoritarios (viajes sin surge) para emparejar la relación a 1:2.
3. **Oversampling (SMOTE):** Generar casos sintéticos de viajes con recargo hasta un ratio 1:2.
4. **Ponderación Algorítmica (`class_weight='balanced'`):** Penalizar fuertemente los errores sobre la clase minoritaria sin alterar el tamaño de la muestra.

> 🛡️ **Garantía Metodológica Antifuga:** Se eliminan categóricamente `price` y `surge_multiplier`. Toda técnica de balanceo se aplica **únicamente en el conjunto de entrenamiento (`Train`)**. El conjunto de prueba (`Test`) permanece 100% fiel a la distribución real.


In [ ]:
# 1. Preparación del subconjunto de Lyft sin Shared
df_lyft = df[(df['cab_type'] == 'Lyft') & (df['name'] != 'Shared')].copy()

# Variables predictoras contextuales (observables antes del viaje)
cols_pred_surge = [
    'source', 'destination', 'hour_local', 'day_of_week', 'is_rush_hour',
    'is_weekend', 'time_slot', 'name', 'distance', 'temperature',
    'precipIntensity', 'humidity', 'windSpeed', 'visibility'
]

X_surge = pd.get_dummies(df_lyft[cols_pred_surge], drop_first=True, dtype=float)
y_surge = df_lyft['has_surge'].values

# Split Estratificado (75% Train, 25% Test)
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_surge, y_surge, test_size=0.25, random_state=42, stratify=y_surge
)

# Estandarización de atributos (fundamental para convergencia de Regresión Logística)
scaler_surge = StandardScaler()
X_tr_s_sc = scaler_surge.fit_transform(X_tr_s)
X_te_s_sc = scaler_surge.transform(X_te_s)

print(f"Dimensiones Train original: {X_tr_s.shape[0]:,} filas | Test original: {X_te_s.shape[0]:,} filas.")


In [ ]:
# 2. Configuración del Laboratorio de Balanceo (Solo en Train)
tecnicas_balanceo = {}

# Técnica 1: Desbalanceado
tecnicas_balanceo['1. Desbalanceado (Base)'] = (X_tr_s_sc, y_tr_s, False)

# Técnica 2: Undersampling (Eliminar clase mayoritaria hasta ratio 0.5)
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_tr_rus, y_tr_rus = rus.fit_resample(X_tr_s_sc, y_tr_s)
tecnicas_balanceo['2. Undersampling (1:2)'] = (X_tr_rus, y_tr_rus, False)

# Técnica 3: SMOTE (Sobremuestreo sintético de la clase minoritaria)
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_tr_smote, y_tr_smote = smote.fit_resample(X_tr_s_sc, y_tr_s)
tecnicas_balanceo['3. SMOTE (1:2)'] = (X_tr_smote, y_tr_smote, False)

# Técnica 4: Cost-Sensitive (Class-Weight Balanced)
tecnicas_balanceo['4. Class-Weight Balanced'] = (X_tr_s_sc, y_tr_s, True)

# Ejecución comparativa y recolección de métricas
resultados_balanceo = []
matrices_confusion = {}

for nombre_tec, (X_train_exp, y_train_exp, use_cw) in tecnicas_balanceo.items():
    cw = 'balanced' if use_cw else None
    
    # A. Regresión Logística
    clf_lr = LogisticRegression(max_iter=1000, class_weight=cw, random_state=42)
    clf_lr.fit(X_train_exp, y_train_exp)
    y_pred_lr = clf_lr.predict(X_te_s_sc)
    y_proba_lr = clf_lr.predict_proba(X_te_s_sc)[:, 1]
    
    # B. Árbol de Decisión
    clf_dt = DecisionTreeClassifier(max_depth=10, class_weight=cw, random_state=42)
    clf_dt.fit(X_train_exp, y_train_exp)
    y_pred_dt = clf_dt.predict(X_te_s_sc)
    y_proba_dt = clf_dt.predict_proba(X_te_s_sc)[:, 1]
    
    # Matrices y métricas
    cm_lr = confusion_matrix(y_te_s, y_pred_lr)
    cm_dt = confusion_matrix(y_te_s, y_pred_dt)
    
    matrices_confusion[f"{nombre_tec}\n(Reg. Logística)"] = cm_lr
    matrices_confusion[f"{nombre_tec}\n(Árbol Decisión)"] = cm_dt
    
    resultados_balanceo.append({
        'Técnica': nombre_tec, 'Modelo': 'Regresión Logística',
        'Recall (Surge)': recall_score(y_te_s, y_pred_lr),
        'Precision': precision_score(y_te_s, y_pred_lr, zero_division=0),
        'F1-Score': f1_score(y_te_s, y_pred_lr),
        'ROC-AUC': roc_auc_score(y_te_s, y_proba_lr)
    })
    resultados_balanceo.append({
        'Técnica': nombre_tec, 'Modelo': 'Árbol Decisión',
        'Recall (Surge)': recall_score(y_te_s, y_pred_dt),
        'Precision': precision_score(y_te_s, y_pred_dt, zero_division=0),
        'F1-Score': f1_score(y_te_s, y_pred_dt),
        'ROC-AUC': roc_auc_score(y_te_s, y_proba_dt)
    })

df_res_balanceo = pd.DataFrame(resultados_balanceo)
print("=== TABLA COMPARATIVA DE TÉCNICAS DE BALANCEO Y MODELOS ===")
display(df_res_balanceo.style.format({
    'Recall (Surge)': '{:.2%}', 'Precision': '{:.2%}', 'F1-Score': '{:.4f}', 'ROC-AUC': '{:.4f}'
}))


In [ ]:
# Visualización de las 8 Matrices de Confusión resultantes
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
titulos = list(matrices_confusion.keys())

for idx, ax in enumerate(axes.flatten()):
    tit = titulos[idx]
    cm = matrices_confusion[tit]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                annot_kws={'size': 11, 'weight': 'bold'})
    ax.set_title(tit, fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicción (0=Normal, 1=Surge)')
    ax.set_ylabel('Real (0=Normal, 1=Surge)')

plt.suptitle("Impacto del Balanceo de Clases en la Detección de Tarifa Dinámica (has_surge)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


---
## 🎯 4. Módulo de Clasificación Multiclase (Uber `service_tier`)
Aprovechando que en Uber las tarifas son cerradas (*Upfront Pricing*) y no desglosan el multiplicador, implementamos la propuesta de **clasificación multiclase**:
- **Objetivo:** Predecir el nivel de servicio contratado (`service_tier`: *Compartido, Estándar, XL, Black, Black XL*) a partir de variables económicas del viaje: `distance`, `price`, `price_per_mile` ($= \text{price}/\text{distance}$) y `straight_line_miles`.
- **Modelos:** **Regresión Logística Multiclase (Softmax)** vs. **Árbol de Decisión Multiclase**.
- **Evaluación:** Matriz de Confusión $5 \times 5$, Macro F1-Score y Accuracy.


In [ ]:
# Preparación del subconjunto de Uber
df_uber = df[df['cab_type'] == 'Uber'].copy()
df_uber['distance_adj'] = df_uber['distance'].clip(lower=0.1)
df_uber['price_per_mile'] = df_uber['price'] / df_uber['distance_adj']

X_uber = df_uber[['distance', 'price', 'price_per_mile', 'straight_line_miles']]
y_uber = df_uber['service_tier']

# Partición Train / Test
X_tr_u, X_te_u, y_tr_u, y_te_u = train_test_split(
    X_uber, y_uber, test_size=0.25, random_state=42, stratify=y_uber
)

# Estandarización
scaler_uber = StandardScaler()
X_tr_u_sc = scaler_uber.fit_transform(X_tr_u)
X_te_u_sc = scaler_uber.transform(X_te_u)

# 1. Regresión Logística Softmax (Multinomial)
clf_u_lr = LogisticRegression(max_iter=1000, random_state=42)
clf_u_lr.fit(X_tr_u_sc, y_tr_u)
y_pred_u_lr = clf_u_lr.predict(X_te_u_sc)

# 2. Árbol de Clasificación Multiclase
clf_u_dt = DecisionTreeClassifier(max_depth=8, random_state=42)
clf_u_dt.fit(X_tr_u, y_tr_u)
y_pred_u_dt = clf_u_dt.predict(X_te_u)

# Métricas globales
acc_u_lr = accuracy_score(y_te_u, y_pred_u_lr)
f1_u_lr = f1_score(y_te_u, y_pred_u_lr, average='macro')

acc_u_dt = accuracy_score(y_te_u, y_pred_u_dt)
f1_u_dt = f1_score(y_te_u, y_pred_u_dt, average='macro')

print(f"-> Regresión Logística Softmax: Accuracy = {acc_u_lr*100:.2f}% | Macro F1 = {f1_u_lr:.4f}")
print(f"-> Árbol Multiclase (depth=8):   Accuracy = {acc_u_dt*100:.2f}% | Macro F1 = {f1_u_dt:.4f}")


In [ ]:
# Matrices de Confusión Multiclase (5x5)
labels_tiers = ['Compartido', 'Estándar', 'XL', 'Black', 'Black XL']
cm_u_lr = confusion_matrix(y_te_u, y_pred_u_lr, labels=labels_tiers)
cm_u_dt = confusion_matrix(y_te_u, y_pred_u_dt, labels=labels_tiers)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm_u_lr, annot=True, fmt='d', cmap='Oranges', xticklabels=labels_tiers, yticklabels=labels_tiers, ax=axes[0])
axes[0].set_title(f"Uber: Regresión Logística Softmax\nAccuracy: {acc_u_lr*100:.2f}% | Macro F1: {f1_u_lr:.4f}", fontweight='bold')
axes[0].set_xlabel("Nivel de Servicio Predicho")
axes[0].set_ylabel("Nivel de Servicio Real")

sns.heatmap(cm_u_dt, annot=True, fmt='d', cmap='Greens', xticklabels=labels_tiers, yticklabels=labels_tiers, ax=axes[1])
axes[1].set_title(f"Uber: Árbol de Decisión Multiclase\nAccuracy: {acc_u_dt*100:.2f}% | Macro F1: {f1_u_dt:.4f}", fontweight='bold')
axes[1].set_xlabel("Nivel de Servicio Predicho")
axes[1].set_ylabel("Nivel de Servicio Real")

plt.tight_layout()
plt.show()


---
## 🎓 5. Conclusiones y Síntesis del Sprint 3

### 1. En Regresión de Precios (`price`):
- El **Árbol de Regresión** ($R^2 = 0.9055$, $\text{RMSE} = \$2.89$) y la **Regresión Lineal Logarítmica** ($R^2 = 0.8875$, $\text{RMSE} = \$3.15$) superan con claridad a la regresión lineal directa.
- Modelar $\log(\text{price})$ reduce el error en viajes de alta gama (Black SUV, Lux Black) estabilizando la heterocedasticidad.

### 2. En el Laboratorio de Balanceo de Clases (`has_surge`):
- En el escenario **Desbalanceado base**, los modelos colapsan a predecir siempre "Sin Surge", obteniendo un **Recall del 0.00% al 1.77%** (inútil para la operación del negocio).
- Aplicar **SMOTE** o **Undersampling** eleva el Recall pero introduce ruido en la frontera.
- La **Ponderación Algorítmica (`class_weight='balanced'`)** demostró ser la estrategia más efectiva: eleva el Recall de detección de tarifas dinámicas hasta el **74.77% en Regresión Logística** y el **83.41% en Árboles**, logrando capturar la gran mayoría de momentos de alta demanda sin descartar datos reales ni inflar sintéticamente la muestra.

### 3. En Clasificación Multiclase de Servicios (Uber):
- La ingeniería de atributos con `price_per_mile` permite separar con altísima nitidez las 5 gamas de Uber, alcanzando un **Accuracy del 86.66% y un Macro F1 de 0.8663** con Árboles de Decisión.
- Las confusiones residuales se concentran lógicamente entre gamas adyacentes (ej. UberX vs. WAV dentro de Estándar), confirmando la coherencia del modelo.
